In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer.IntervalTrainer import IntervalTrainer
from src.trainer.SimpleTrainer import SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils import InContextHead
from src import models
from src.regulariser.UnbiasRegulariser import UnbiasRegulariser
from src.regulariser.MultiRegulariser import MultiRegulariser
from src.regulariser.L2Regulariser import L2Regulariser

### Task Incremental Learning

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cpu")
head.set_context(0)
model = models.get_fully_connected_mnist_model(head=head)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Train the model on the original task.

In [ ]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Train subsequent tasks with bounds.

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.9,
    primal_learning_rate=0.5,
    paradigm="TIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0], context_id=0)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, context_id=i)
    interval_trainer.test(test_tasks[0 : i + 1], context_list=list(range(i + 1)))
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test, context_id=i)

### Domain Incremental Training

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [5]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [6]:
l2_reg = L2Regulariser(lmbd=0.01)
ubias_reg = UnbiasRegulariser(lmbd=0.01)
regulariser = MultiRegulariser([l2_reg, ubias_reg])

trainer = SimpleTrainer(model, domain_map_fn=domain_map_fn)
trainer.train(
    train_tasks[0],
    val_tasks[0],
    epochs=3,
)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|██████████| 3/3 [00:47<00:00, 15.92s/it, val_loss=0.00845, val_acc=0.997]


Test Results: [(0.0103, 0.9968)]


In [8]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.9,
    primal_learning_rate=0.33,
    dual_learning_rate=0.01,
    batch_size=400,
    paradigm="DIL",
    domain_map_fn=domain_map_fn
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:2], val_tasks[1:2], test_tasks[1:2])
):
    interval_trainer.train(train, val, batch_size=256)
    interval_trainer.test(test_tasks[0 : i + 2])
    # if i < len(train_tasks) - 2:
    #     interval_trainer.compute_rashomon_set(test, domain_map_fn=domain_map_fn)

---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0961 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.90
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=1.00,  Min acc soft=1.00


100%|██████████| 200/200 [11:04<00:00,  3.32s/it, size=1081.94, obj=0.175, min_soft_acc=0.869]   


Final bbox:  Obj=0.18,  Size=1081.94,  Min acc hard=0.89,  Min acc soft=0.88
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['35.23', '184.74', '393.39', '580.45', '722.80', '837.62', '927.82', '994.54', '1046.79', '1081.94']
Checkpoint certificates: ['0.92', '0.91', '0.92', '0.91', '0.89', '0.89', '0.89', '0.89', '0.89', '0.89']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|██████████| 5/5 [01:15<00:00, 15.18s/it, loss=1.9040, val_acc=0.4388, proj=1]


Test Results: [(0.0139, 0.9956), (1.759, 0.4464)]


### Class Incremental Learning

In [9]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [10]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|██████████| 3/3 [00:46<00:00, 15.41s/it, val_loss=0.0131, val_acc=0.996]


Test Results: [(0.0141, 0.9959)]


In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.8,
    primal_learning_rate=0.5,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:])
):
    interval_trainer.train(train, val, batch_size=256)
    interval_trainer.test(test_tasks[0 : i + 2])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)

---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1961 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=1.00,  Min acc soft=1.00


100%|██████████| 300/300 [16:51<00:00,  3.37s/it, size=2415.11, obj=0.390, min_soft_acc=0.762]   


Final bbox:  Obj=0.39,  Size=2415.11,  Min acc hard=0.79,  Min acc soft=0.79
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 15 checkpoints
Checkpoints sizes: ['44.60', '266.56', '527.39', '710.76', '900.06', '1076.23', '1246.38', '1421.47', '1574.77', '1742.00', '1901.23', '2031.13', '2161.57', '2273.52', '2415.11']
Checkpoint certificates: ['0.93', '0.79', '0.82', '0.79', '0.77', '0.81', '0.80', '0.79', '0.80', '0.80', '0.80', '0.80', '0.79', '0.80', '0.79']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

### Class Incremental Learning + Regulariser

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
regulariser = UnbiasRegulariser(lmbd=0.01)

trainer = SimpleTrainer(model)
trainer.train(
    train_tasks[0], val_tasks[0], epochs=3, batch_size=256, regulariser=regulariser
)
trainer.test(test_tasks[0:1])

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.8,
    primal_learning_rate=0.5,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:])
):
    interval_trainer.train(train, val, batch_size=256, regulariser=regulariser)
    interval_trainer.test(test_tasks[0 : i + 2])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)